# Exploratory statistical significance for a matrix profile

A matrix profile reports nearest-neighbor distances, but a small distance is not automatically evidence that a motif is meaningful. This tutorial shows one reproducible, exploratory way to compare an observed minimum profile distance with a null distribution made from circular block permutations. It is a teaching aid, not a universal significance cutoff.

## Set up a series with a repeated motif

The fixed random seed makes the example reproducible. The injected waveform is repeated at two known locations, while the rest of the series contains a smooth background and noise.

In [ ]:
%matplotlib inline
import matplotlib.pyplot as plt
import numpy as np
import stumpy

rng = np.random.default_rng(2025)
n, m = 240, 24
time = np.arange(n)
series = 0.35 * np.sin(time / 18) + 0.08 * rng.normal(size=n)
motif = 0.8 * np.sin(np.linspace(0, 2 * np.pi, m))
for start in (58, 154):
    series[start : start + m] += motif

plt.figure(figsize=(12, 3))
plt.plot(series, color="#3366aa")
plt.axvspan(58, 58 + m, alpha=0.2, color="#dd8844")
plt.axvspan(154, 154 + m, alpha=0.2, color="#dd8844")
plt.title("Synthetic series with a repeated motif")
plt.xlabel("Sample")
plt.show()

## Measure the observed motif distance

We use the minimum finite matrix-profile value as a deliberately simple test statistic. The profile index gives the location of the strongest candidate, but the value alone does not provide a p-value.

In [ ]:
profile = stumpy.stump(series, m)[:, 0]
finite_profile = profile[np.isfinite(profile)]
observed_distance = float(np.min(finite_profile))
observed_index = int(np.nanargmin(profile))
print(f"candidate subsequence: {observed_index}")
print(f"observed minimum distance: {observed_distance:.4f}")

## Build a block-permutation null distribution

Independent shuffling would destroy all local dependence. Instead, we split the series into circular blocks, rotate the block order, and rotate each block internally. This preserves short-range structure approximately while breaking the original motif alignment. The block size is a modeling choice, not a theorem.

In [ ]:
def circular_block_permutation(values, block_size, random_generator):
    """Return one reproducible circular block permutation."""
    blocks = [values[i : i + block_size] for i in range(0, len(values), block_size)]
    random_generator.shuffle(blocks)
    rotated = [np.roll(block, random_generator.integers(len(block))) for block in blocks]
    return np.concatenate(rotated)[: len(values)]

block_size = m
n_surrogates = 24
surrogate_minima = np.empty(n_surrogates)
for i in range(n_surrogates):
    surrogate = circular_block_permutation(series, block_size, rng)
    surrogate_profile = stumpy.stump(surrogate, m)[:, 0]
    surrogate_minima[i] = np.nanmin(surrogate_profile)

# +1 avoids reporting an exact zero from a small finite sample.
p_value = (1 + np.count_nonzero(surrogate_minima <= observed_distance)) / (n_surrogates + 1)
percentile = 100 * (1 - p_value)
print(f"empirical lower-tail p-value: {p_value:.3f}")
print(f"observed distance is above {percentile:.1f}% of the null minima")

In [ ]:
plt.figure(figsize=(8, 3))
plt.hist(surrogate_minima, bins=8, color="#99bbee", edgecolor="white")
plt.axvline(observed_distance, color="#cc5533", linewidth=2, label="observed minimum")
plt.xlabel("minimum matrix-profile distance")
plt.ylabel("surrogate count")
plt.title("Observed statistic against the block-permutation null")
plt.legend()
plt.show()

## Check sensitivity to the number of surrogates

The estimate is intentionally coarse with 24 surrogates. Repeating the calculation with more surrogates should make the histogram and percentile less variable, at a higher runtime cost. For a real analysis, choose the count before inspecting the result and report the seed, block size, and number of surrogates.

## Interpretation and limitations

* Overlapping subsequences are dependent, so this is not a collection of independent tests.
* Searching every profile index is a multiple-comparisons problem; using the minimum statistic partly reflects that search but does not solve every inference issue.
* The null distribution depends on the block size and permutation scheme. Different choices answer different questions.
* The empirical p-value is a descriptive comparison, not a guarantee that the motif is causal, novel, or useful. Validate any threshold on domain-specific data.